# Homework 2: Securing the AutoDrive Firmware Pipeline

## Elaborado por

- Pablo Alvarado / 00344965
- Andres Proano / 00327809

## Notebook tecnico y academico

**Caso de estudio.** AutoDrive distribuye actualizaciones de firmware a una flota de vehiculos autonomos. El objetivo de este notebook es justificar, paso a paso, como la criptografia permite que cada actualizacion sea autentica, integra, confidencial y trazable.

**Enfoque metodologico.** El desarrollo combina teoria criptografica, calculos manuales, pequenas comprobaciones en Python y decisiones de diseno conectadas directamente con el caso AutoDrive.

## Estructura propuesta del notebook

**Objetivo.** Presentar una ruta de lectura clara, continua y facil de evaluar.

1. Introduccion general al problema de seguridad en AutoDrive.
2. Parte 1. Generacion manual de claves RSA y verificacion del esquema.
3. Parte 2. Uso de SHA-256 para integridad y deteccion de manipulacion.
4. Parte 3. Certificados digitales y cadena de confianza basada en X.509.
5. Parte 4. Firmas digitales y no repudio en el envio del firmware.
6. Parte 5. Diseno de un protocolo hibrido de actualizacion segura.
7. Conclusion final.
8. Referencias tecnicas.

Esta estructura hace que el notebook se lea como un informe tecnico: primero se construyen los fundamentos, luego se justifica la cadena de confianza y finalmente se integra todo en un protocolo completo para AutoDrive.

## Introduccion

**Objetivo.** Explicar por que la distribucion segura de firmware exige combinar varios mecanismos criptograficos y no una sola tecnica aislada.

En el escenario de AutoDrive, una actualizacion de firmware no es solo un archivo que viaja por la red. Es una instruccion critica que puede modificar el comportamiento de un vehiculo autonomo. Por ello, el sistema debe responder simultaneamente a cinco preguntas de seguridad:

- Como se protege el contenido del firmware frente a espionaje.
- Como se detecta cualquier modificacion durante el transporte.
- Como se verifica que el emisor verdadero es AutoDrive HQ.
- Como se prueba que una version concreta fue autorizada oficialmente.
- Como se escala el proceso cuando el firmware es grande y la flota es numerosa.

La respuesta correcta no es usar RSA para todo ni cifrar archivos gigantes con criptografia asimetrica. La solucion realista es una arquitectura por capas: RSA o un esquema equivalente para proteger claves pequenas, SHA-256 para resumir archivos grandes, certificados X.509 para establecer confianza en la clave publica correcta y firmas digitales para demostrar autorizacion. La Parte 5 integra esos componentes en un protocolo de actualizacion segura para el caso AutoDrive.

## Parte 1. RSA Key Generation

**Objetivo.** Construir manualmente un ejemplo pequeno de RSA y usarlo para explicar por que el mismo principio sigue siendo seguro cuando los numeros crecen hasta el tamano real de produccion.

### Explicacion teorica

RSA parte de dos primos $p$ y $q$. A partir de ellos se construye el modulo

$$
n = p \cdot q
$$

junto con la funcion totiente de Euler

$$
\phi(n) = (p-1)(q-1).
$$

Luego se elige un exponente publico $e$ tal que

$$
\gcd(e, \phi(n)) = 1,
$$

porque solo asi existe un inverso multiplicativo modulo $\phi(n)$. Ese inverso es el exponente privado $d$, definido por

$$
ed \equiv 1 \pmod{\phi(n)}.
$$

La clave publica es el par $(n,e)$ y la clave privada es el par $(n,d)$. En un despliegue real, el vehiculo podria usar esta logica para proteger material secreto pequeno, por ejemplo una clave de sesion enviada por AutoDrive HQ.

### Desarrollo paso a paso

Se nos pide usar los primos pequenos $p=11$ y $q=13$.

1. Calculo del modulo $n$.

$$
n = 11 \cdot 13 = 143
$$

2. Calculo de la funcion totiente.

$$
\phi(n) = (11-1)(13-1) = 10 \cdot 12 = 120
$$

3. Eleccion del exponente publico.
Buscamos un valor coprimo con $120$. Elegimos

$$
e = 7
$$

porque $\gcd(7,120)=1$.

4. Calculo del exponente privado.
Necesitamos resolver

$$
7d \equiv 1 \pmod{120}.
$$

Aplicando el algoritmo extendido de Euclides:

$$
120 = 17 \cdot 7 + 1
$$

y por tanto

$$
1 = 120 - 17 \cdot 7.
$$

Esto muestra que

$$
7(-17) \equiv 1 \pmod{120}.
$$

Luego,

$$
d \equiv -17 \equiv 103 \pmod{120},
$$

asi que tomamos

$$
d = 103.
$$

5. Claves obtenidas.

- **Clave publica:** $(n,e) = (143,7)$
- **Clave privada:** $(n,d) = (143,103)$

### Verificacion manual con un mensaje pequeno

Tomemos el mensaje $m=5$.

**Cifrado**

$$
c = m^e \bmod n = 5^7 \bmod 143.
$$

Calculamos por potencias:

- $5^2 = 25$
- $5^4 = 25^2 = 625 \equiv 53 \pmod{143}$
- $5^7 = 5^4 \cdot 5^2 \cdot 5 \equiv 53 \cdot 25 \cdot 5 = 6625 \equiv 47 \pmod{143}$

Por tanto,

$$
c = 47.
$$

**Descifrado**

$$
m = c^d \bmod n = 47^{103} \bmod 143.
$$

Comprobamos primero modulo $11$:

$$
47 \equiv 3 \pmod{11}, \qquad 103 \equiv 3 \pmod{10},
$$

de modo que

$$
47^{103} \equiv 3^3 = 27 \equiv 5 \pmod{11}.
$$

Comprobamos ahora modulo $13$:

$$
47 \equiv 8 \pmod{13}, \qquad 103 \equiv 7 \pmod{12},
$$

de modo que

$$
47^{103} \equiv 8^7 \equiv 8^4 \cdot 8^2 \cdot 8 \equiv 1 \cdot 12 \cdot 8 = 96 \equiv 5 \pmod{13}.
$$

El resultado es congruente con $5$ tanto modulo $11$ como modulo $13$, asi que por el Teorema Chino del Resto el mensaje recuperado es

$$
m = 5.
$$

### Mini conclusion

El ejemplo confirma la mecanica central de RSA: la clave publica sirve para cifrar y la clave privada para revertir ese proceso. Aunque este ejercicio usa numeros pequenos para poder desarrollarlo a mano, la misma estructura algebraica se emplea en sistemas reales.

In [1]:
from math import gcd

# Parametros del ejemplo manual
p = 11
q = 13
e = 7
m = 5

n = p * q
phi = (p - 1) * (q - 1)
assert gcd(e, phi) == 1, 'e debe ser coprimo con phi(n)'

d = pow(e, -1, phi)
c = pow(m, e, n)
m_recovered = pow(c, d, n)

print(f'p = {p}, q = {q}')
print(f'n = {n}')
print(f'phi(n) = {phi}')
print(f'e = {e}')
print(f'd = {d}')
print(f'Clave publica = ({n}, {e})')
print(f'Clave privada = ({n}, {d})')
print(f'Mensaje original = {m}')
print(f'Cifrado = {c}')
print(f'Mensaje recuperado = {m_recovered}')

p = 11, q = 13
n = 143
phi(n) = 120
e = 7
d = 103
Clave publica = (143, 7)
Clave privada = (143, 103)
Mensaje original = 5
Cifrado = 47
Mensaje recuperado = 5


### Por que en RSA real de 2048 bits no es viable hallar $d$ a partir de $n$ y $e$

**Objetivo.** Conectar el ejemplo pequeno con la seguridad practica de RSA en tamano real.

En este ejercicio fue sencillo hallar $d$ porque conocer $p=11$ y $q=13$ permite calcular de inmediato

$$
\phi(n) = (p-1)(q-1).
$$

Una vez conocido $\phi(n)$, obtener $d$ es solo un problema algebraico: calcular el inverso modular de $e$ modulo $\phi(n)$, es decir,

$$
d \equiv e^{-1} \pmod{\phi(n)}.
$$

El problema real para un atacante aparece antes. Si solo conoce la clave publica $(n,e)$, necesita recuperar $\phi(n)$ para calcular $d$. Pero para obtener $\phi(n)$ en RSA clasico debe factorizar

$$
n = p \cdot q
$$

y descubrir los primos secretos $p$ y $q$.

En un sistema RSA de 2048 bits, $n$ es el producto de dos primos enormes, tipicamente de alrededor de 1024 bits cada uno. No se conoce un algoritmo clasico eficiente que permita factorizar de manera practica un semiprimo de ese tamano. Por eso, aunque la relacion matematica entre $e$, $d$ y $\phi(n)$ es simple, la recuperacion de $d$ a partir de la clave publica se vuelve computacionalmente inviable.

En otras palabras, la dificultad no esta en invertir la congruencia

$$
ed \equiv 1 \pmod{\phi(n)},
$$

sino en que el atacante no conoce $\phi(n)$. Y no puede conocer $\phi(n)$ sin factorizar $n$ primero. Esa es la conexion esencial entre la seguridad de RSA y la dureza computacional de la factorizacion.

### Mini conclusion

RSA sigue siendo util porque transforma una operacion facil para el propietario de la clave privada en un problema extremadamente dificil para un atacante externo. En AutoDrive, esto permite usar criptografia asimetrica para proteger secretos pequenos, como una clave de sesion, sin exponer la clave privada del vehiculo.

## Parte 2. Cryptographic Hashes

**Objetivo.** Explicar por que SHA-256 es el mecanismo adecuado para verificar integridad en archivos de firmware grandes y por que un cambio minimo debe producir una salida radicalmente distinta.

### Explicacion teorica

SHA-256 es una funcion hash criptografica que transforma una entrada de longitud arbitraria en una salida fija de 256 bits. En el problema de AutoDrive, su papel no es cifrar el firmware, sino producir un resumen corto y estable del archivo para poder verificar su integridad sin operar directamente sobre los 2 GB en cada paso de verificacion.

Si denotamos el firmware por $M$, su resumen criptografico es

$$
h = \text{SHA-256}(M).
$$

Cuando AutoDrive HQ calcula el hash del firmware original, obtiene una huella digital unica para esa version. Si el vehiculo recibe el archivo y calcula exactamente el mismo hash, eso constituye evidencia de que el contenido recibido coincide bit a bit con el contenido esperado.

### Avalanche effect

El **avalanche effect** significa que un cambio minusculo en la entrada, incluso un solo bit, debe provocar un cambio muy grande en la salida. Si un atacante altera un bit y obtenemos $M'$, el nuevo resumen es

$$
h' = \text{SHA-256}(M').
$$

Lo esperado es que $h$ y $h'$ difieran en aproximadamente la mitad de sus bits. En terminos de distancia de Hamming, lo expresamos como

$$
\operatorname{HD}(h, h') \approx 128 \text{ bits de } 256.
$$

Esta propiedad es crucial frente a un ataque Man in the Middle. Si un atacante intercepta la actualizacion y altera una parte del firmware, no deberia poder conservar un hash parecido. El resumen nuevo sera radicalmente distinto y la verificacion fallara.

### Colisiones

Una **colision** ocurre cuando dos entradas diferentes producen exactamente el mismo hash. Si un atacante consiguiera que un archivo malicioso `Virus.bin` y el archivo legitimo `Update.bin` tuvieran el mismo SHA-256, el sistema de verificacion quedaria comprometido: el vehiculo podria aceptar el archivo malicioso porque el resumen aparente coincide con el esperado.

Por eso la resistencia a colisiones es un requisito central. La seguridad del flujo de actualizacion depende de que el hash identifique de manera confiable una version concreta del firmware.

### Desarrollo paso a paso

1. Se construye un bloque binario simulado que representa una pequena porcion del firmware.
2. Se calcula su SHA-256 original.
3. Se modifica un solo bit del bloque.
4. Se recalcula el hash.
5. Se comparan ambos resumenes bit a bit.
6. Se cuenta cuantos bits cambiaron.

### Mini conclusion

SHA-256 no reemplaza a la firma digital, pero la hace eficiente: en lugar de operar sobre un archivo enorme para autenticarlo, se autentica su resumen. En AutoDrive, esto reduce costo computacional y fortalece la deteccion de manipulacion.

In [2]:
import hashlib

# Bloque binario simulado de firmware
original = bytearray((i * 37) % 256 for i in range(1024))
modified = bytearray(original)

# Cambiamos exactamente un bit
byte_index = 100
bit_mask = 0b00010000
modified[byte_index] ^= bit_mask


def sha256_digest(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()


def count_changed_bits(a: bytes, b: bytes) -> int:
    return sum((x ^ y).bit_count() for x, y in zip(a, b))


digest_original = sha256_digest(original)
digest_modified = sha256_digest(modified)
changed_bits = count_changed_bits(digest_original, digest_modified)
total_bits = len(digest_original) * 8
percentage = (changed_bits / total_bits) * 100

print(f'SHA-256 original : {digest_original.hex()}')
print(f'SHA-256 modificado: {digest_modified.hex()}')
print(f'Bits diferentes  : {changed_bits} de {total_bits}')
print(f'Porcentaje       : {percentage:.2f}%')

SHA-256 original : 85c1a4ba1d2345309bd0038e835f0536a88851fbd14c7ed266b958bd5180898f
SHA-256 modificado: 7c3d8ef3e587cd7a00f684f189cbefa6dafbc597a222c3daf5bce31ec7b74882
Bits diferentes  : 126 de 256
Porcentaje       : 49.22%


### Lectura de la comprobacion

La salida del experimento no tiene por que mostrar exactamente 128 bits cambiados en cada ejecucion, porque el comportamiento esperado es estadistico, no determinista. Sin embargo, si SHA-256 presenta correctamente el avalanche effect, la cantidad de bits alterados deberia quedar cerca de la mitad del digest total.

### Mini conclusion

Para AutoDrive, esta propiedad significa que una alteracion pequena en el firmware no puede pasar inadvertida como si fuera un cambio menor. El resumen criptografico reacciona de forma desproporcionada y hace visible la manipulacion.

### Demostracion practica: busqueda de colision en un hash truncado

Para ilustrar el concepto de colision, truncamos SHA-256 a sus primeros 16 bits (2 bytes). Con solo $2^{16}=65\,536$ valores posibles, el Birthday Paradox predice que encontraremos una colision tras aproximadamente $2^{8}=256$ intentos. Esto muestra por que SHA-256 necesita sus 256 bits completos para ser resistente a colisiones.

In [3]:
import hashlib, os

TRUNC_BYTES = 2  # 16 bits

seen: dict[bytes, bytes] = {}
attempts = 0

while True:
    data = os.urandom(32)
    attempts += 1
    short_hash = hashlib.sha256(data).digest()[:TRUNC_BYTES]

    if short_hash in seen and seen[short_hash] != data:
        collider = seen[short_hash]
        break
    seen[short_hash] = data

print(f'Colision encontrada tras {attempts} intentos (hash truncado a {TRUNC_BYTES * 8} bits)\n')
print(f'Input A : {collider.hex()}')
print(f'Input B : {data.hex()}')
print(f'SHA-256 A (completo): {hashlib.sha256(collider).hexdigest()}')
print(f'SHA-256 B (completo): {hashlib.sha256(data).hexdigest()}')
print(f'Primeros {TRUNC_BYTES * 8} bits    : coinciden  ->  {short_hash.hex()}')
print(f'Hash completo        : totalmente diferente')
print(f'\nConclusion: con 256 bits completos, este ataque necesitaria ~2^128 intentos, algo inviable.')

Colision encontrada tras 185 intentos (hash truncado a 16 bits)

Input A : a93740babaee503c64e11f82ec9f409c7aa2fe1666e6b873a1709ff6b0462805
Input B : ce41f7f47e66a754f28097b47df3061753e1b0a08b1230c4fd7eefaf4b73e90d
SHA-256 A (completo): 1581c343b034e596bcc105a346c6337c9f3003ad5102c94027899f5a06f6790f
SHA-256 B (completo): 1581fd53e67ca893d19e7f475dfd115ff796a37b8e16af824c5f4aaf8577ca9c
Primeros 16 bits    : coinciden  ->  1581
Hash completo        : totalmente diferente

Conclusion: con 256 bits completos, este ataque necesitaria ~2^128 intentos, algo inviable.


## Parte 3. Digital Certificates

**Objetivo.** Justificar por que el vehiculo no debe confiar ciegamente en una clave publica enviada en texto plano y explicar como los certificados X.509 resuelven ese problema mediante una cadena de confianza.

### Explicacion teorica

No basta con enviar la clave publica de HQ en un archivo de texto porque ese archivo no demuestra ninguna vinculacion autentica entre la clave y la identidad de AutoDrive HQ. Un atacante ubicado en el medio del canal podria sustituir la clave publica legitima por su propia clave y hacer que el vehiculo verifique firmas falsas o cifre secretos para el atacante. En ese escenario, el problema no es la criptografia en si, sino la **autenticidad de la clave publica recibida**.

Un certificado X.509 resuelve este punto porque vincula una identidad con una clave publica y protege esa vinculacion mediante la firma digital de una Autoridad de Certificacion (CA). El vehiculo ya no necesita confiar en un archivo cualquiera; necesita validar una cadena de certificados hasta una raiz de confianza conocida.

En notacion resumida, la cadena que el vehiculo espera validar es

$$
\mathrm{Cert}_{HQ} \to \mathrm{CA}_{intermedia} \to \mathrm{RootCA}_{confiable}.
$$

### Rol del certificado X.509

Un certificado X.509 cumple tres funciones practicas en AutoDrive:

1. Identifica al sujeto del certificado, por ejemplo AutoDrive HQ.
2. Transporta la clave publica que se utilizara para verificar firmas o establecer sesiones seguras.
3. Incluye una firma de la CA que afirma que esa clave publica pertenece realmente a esa identidad, dentro de un periodo de validez y bajo ciertas extensiones o restricciones.

### Proceso CSR

Un **Certificate Signing Request (CSR)** es la solicitud formal que AutoDrive HQ envia a una CA para pedir la emision de un certificado. En terminos academicos, el proceso es el siguiente:

1. HQ genera localmente su par de claves.
2. HQ construye una solicitud que contiene su identidad, su clave publica y atributos relevantes.
3. HQ firma esa solicitud con su propia clave privada para demostrar posesion de la clave correspondiente.
4. La CA valida la informacion presentada y, si la considera correcta, emite un certificado X.509 firmado.

### Cuatro elementos principales que HQ debe enviar o incluir

Los cuatro componentes principales del paquete de solicitud son:

1. **Subject Distinguished Name (DN).** Identidad del solicitante, por ejemplo organizacion, nombre comun y otros campos distintivos.
2. **Subject Public Key Info.** La clave publica que quedara certificada.
3. **Atributos o extensiones solicitadas.** Por ejemplo `subjectAltName`, `keyUsage` o `extendedKeyUsage`, segun el perfil del certificado requerido.
4. **Firma digital del CSR.** Prueba de posesion de la clave privada asociada a la clave publica incluida en la solicitud.

Ademas del contenido criptografico minimo del CSR, la CA puede pedir evidencia administrativa adicional, como validacion de dominio u organizacion. Esa evidencia complementa el proceso, pero no reemplaza la estructura criptografica del CSR.

### Root certificate que debe estar preinstalado en el vehiculo

El vehiculo debe tener preinstalado en fabrica el **certificado raiz de la CA de confianza** que firma directamente el certificado de HQ o que aparece como ancla de confianza al inicio de la cadena de certificacion. Dicho de forma precisa: el vehiculo debe confiar en la Root CA que firma o encadena hacia el certificado de HQ.

Si la CA emisora real es una CA intermedia, el vehiculo no necesita confiar ciegamente en esa intermedia por si sola; lo esencial es que la cadena presentada por HQ termine en una raiz confiable ya instalada en el vehiculo como ancla de confianza.

### Mini conclusion

Los certificados digitales no agregan seguridad por decorar una clave publica, sino por resolver el problema de distribucion autentica de claves. En AutoDrive, esa cadena de confianza es indispensable para que el vehiculo sepa que la clave usada para verificar una actualizacion pertenece realmente a AutoDrive HQ.

### Demostracion practica: generacion de Root CA, CSR y certificado X.509

El siguiente codigo reproduce el flujo completo de certificacion para AutoDrive:
1. Genera una Root CA autofirmada (la que se instala en fabrica en cada vehiculo).
2. AutoDrive HQ genera su par de claves y construye un CSR con los cuatro elementos descritos arriba.
3. La Root CA firma el CSR y emite un certificado X.509 para HQ.
4. Se inspeccionan los campos del certificado resultante.

In [4]:
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa
import datetime

# ── 1. Root CA (se instala en el vehiculo en fabrica) ──
ca_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
ca_name = x509.Name([
    x509.NameAttribute(NameOID.COUNTRY_NAME, 'US'),
    x509.NameAttribute(NameOID.ORGANIZATION_NAME, 'AutoDrive Trusted Root CA'),
    x509.NameAttribute(NameOID.COMMON_NAME, 'AutoDrive Root CA'),
])
ca_cert = (
    x509.CertificateBuilder()
    .subject_name(ca_name)
    .issuer_name(ca_name)
    .public_key(ca_key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(datetime.datetime.now(datetime.timezone.utc))
    .not_valid_after(datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=3650))
    .add_extension(x509.BasicConstraints(ca=True, path_length=None), critical=True)
    .sign(ca_key, hashes.SHA256())
)
print('=== ROOT CA CERTIFICATE ===')
print(f'Subject  : {ca_cert.subject}')
print(f'Issuer   : {ca_cert.issuer}')
print(f'Valid    : {ca_cert.not_valid_before_utc} -> {ca_cert.not_valid_after_utc}')
print(f'Serial   : {ca_cert.serial_number}')

# ── 2. AutoDrive HQ genera su clave y construye un CSR ──
hq_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
csr = (
    x509.CertificateSigningRequestBuilder()
    .subject_name(x509.Name([
        x509.NameAttribute(NameOID.COUNTRY_NAME, 'US'),
        x509.NameAttribute(NameOID.STATE_OR_PROVINCE_NAME, 'California'),
        x509.NameAttribute(NameOID.ORGANIZATION_NAME, 'AutoDrive Inc.'),
        x509.NameAttribute(NameOID.COMMON_NAME, 'firmware.autodrive.com'),
    ]))
    .add_extension(
        x509.SubjectAlternativeName([x509.DNSName('firmware.autodrive.com')]),
        critical=False,
    )
    .sign(hq_key, hashes.SHA256())  # firma del CSR = prueba de posesion
)

print('\n=== CERTIFICATE SIGNING REQUEST (CSR) ===')
print(f'Subject         : {csr.subject}')
print(f'Firma valida    : {csr.is_signature_valid}')
print(f'Algoritmo firma : {csr.signature_algorithm_oid._name}')
print('Extensiones     :', [type(e.value).__name__ for e in csr.extensions])

# ── 3. La Root CA firma el CSR y emite el certificado de HQ ──
hq_cert = (
    x509.CertificateBuilder()
    .subject_name(csr.subject)
    .issuer_name(ca_cert.subject)
    .public_key(csr.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(datetime.datetime.now(datetime.timezone.utc))
    .not_valid_after(datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=365))
    .add_extension(x509.BasicConstraints(ca=False, path_length=None), critical=True)
    .add_extension(
        x509.KeyUsage(
            digital_signature=True, key_encipherment=False, content_commitment=False,
            data_encipherment=False, key_agreement=False, key_cert_sign=False,
            crl_sign=False, encipher_only=False, decipher_only=False,
        ),
        critical=True,
    )
    .sign(ca_key, hashes.SHA256())
)

print('\n=== HQ CERTIFICATE (emitido por Root CA) ===')
print(f'Subject  : {hq_cert.subject}')
print(f'Issuer   : {hq_cert.issuer}')
print(f'Valid    : {hq_cert.not_valid_before_utc} -> {hq_cert.not_valid_after_utc}')
print(f'Key Usage: {hq_cert.extensions.get_extension_for_class(x509.KeyUsage).value}')
print(f'\nEl vehiculo valida este certificado verificando que el issuer coincide')
print(f'con el subject de la Root CA preinstalada y que la firma es correcta.')

=== ROOT CA CERTIFICATE ===
Subject  : <Name(C=US,O=AutoDrive Trusted Root CA,CN=AutoDrive Root CA)>
Issuer   : <Name(C=US,O=AutoDrive Trusted Root CA,CN=AutoDrive Root CA)>
Valid    : 2026-04-07 03:41:40+00:00 -> 2036-04-04 03:41:40+00:00
Serial   : 52883375352535212702277411907580829186421518824

=== CERTIFICATE SIGNING REQUEST (CSR) ===
Subject         : <Name(C=US,ST=California,O=AutoDrive Inc.,CN=firmware.autodrive.com)>
Firma valida    : True
Algoritmo firma : sha256WithRSAEncryption
Extensiones     : ['SubjectAlternativeName']

=== HQ CERTIFICATE (emitido por Root CA) ===
Subject  : <Name(C=US,ST=California,O=AutoDrive Inc.,CN=firmware.autodrive.com)>
Issuer   : <Name(C=US,O=AutoDrive Trusted Root CA,CN=AutoDrive Root CA)>
Valid    : 2026-04-07 03:41:40+00:00 -> 2027-04-07 03:41:40+00:00
Key Usage: <KeyUsage(digital_signature=True, content_commitment=False, key_encipherment=False, data_encipherment=False, key_agreement=False, key_cert_sign=False, crl_sign=False, encipher_only=Fa

## Parte 4. Digital Signatures

**Objetivo.** Describir con precision el flujo de firma del firmware y explicar por que la firma digital aporta autenticacion y no repudio en el caso AutoDrive.

### Explicacion teorica

El flujo correcto de firma digital no consiste en cifrar todo el archivo de firmware con RSA. Eso seria ineficiente y conceptualmente incorrecto para archivos grandes. El procedimiento correcto es:

1. AutoDrive HQ calcula un hash criptografico del firmware o, de forma aun mas robusta, de un manifiesto que incluye el hash del firmware y metadatos de la version.
2. HQ aplica su clave privada al proceso de firma digital sobre ese resumen, usando un algoritmo de firma adecuado.
3. El vehiculo valida el certificado de HQ, extrae de alli la clave publica legitima y verifica la firma.
4. El vehiculo vuelve a calcular el hash localmente y comprueba que coincide con el valor autenticado por la firma.

Si denotamos por $M_f$ el manifiesto que describe la actualizacion, el flujo matematico minimo puede resumirse como

$$
h = \text{SHA-256}(M_f), \qquad \sigma = \mathrm{Sign}_{sk_{HQ}}(h).
$$

Luego, el vehiculo acepta la autorizacion solo si

$$
\mathrm{Verify}_{pk_{HQ}}(h, \sigma) = \text{valida}.
$$

### Que se hashea

Lo esencial es hashear la representacion exacta de la version que se desea autorizar. En una implementacion minima, se hashea el archivo `Update.bin`. En una implementacion mas solida, se firma un manifiesto que contiene como minimo:

- version del firmware
- hash SHA-256 del archivo
- identificador del modelo o familia de vehiculos
- fecha, numero de compilacion o politica de validez

Firmar un manifiesto es preferible porque no solo autentica los bytes del archivo, sino tambien el contexto de lanzamiento de esa version especifica.

### Que clave se usa

- La firma se genera con la **clave privada de HQ**.
- La firma se verifica con la **clave publica de HQ** obtenida del certificado X.509 validado.

Desde un punto de vista pedagogico, a veces se dice que la firma "cifra el hash con la clave privada". La idea ayuda a recordar que solo el propietario de la clave privada puede producir una firma valida. Sin embargo, tecnicamente es mas preciso decir que el algoritmo de firma aplica una operacion criptografica sobre un resumen codificado y que la verificacion posterior se realiza con la clave publica correspondiente.

### Non repudiation en AutoDrive

El **non repudiation** o no repudio significa que HQ no puede negar de forma creible haber autorizado una version determinada si existe una firma valida producida con su clave privada y asociada a su identidad certificada. En el contexto legal y tecnico del caso AutoDrive, esto aporta dos tipos de evidencia:

1. **Evidencia tecnica.** La firma valida demuestra que la clave privada de HQ fue usada para autorizar ese paquete o manifiesto.
2. **Evidencia de atribucion.** El certificado vincula esa clave con la identidad institucional de AutoDrive HQ dentro de una cadena de confianza aceptada.

Si un firmware defectuoso causa un incidente, la firma digital permite relacionar una version concreta con una autorizacion concreta. Esto no reemplaza por si solo un proceso legal completo, pero proporciona una base tecnica fuerte para demostrar que esa version fue emitida o aprobada desde el entorno de claves de AutoDrive HQ.

### Mini conclusion

La firma digital no solo dice que el archivo no cambio; dice ademas que una entidad autentica y trazable autorizo esa version especifica. Esa diferencia es fundamental en un entorno critico como el de vehiculos autonomos.

### Demostracion practica: firma digital del firmware y caso de fallo

El siguiente codigo simula el flujo completo de firma:
1. HQ firma el hash de un firmware simulado con su clave privada.
2. El vehiculo verifica la firma con la clave publica de HQ.
3. Se modifica un byte del firmware y se muestra como la verificacion falla.

In [5]:
from cryptography.hazmat.primitives.asymmetric import padding, utils
import hashlib, os, json

# Reutilizamos hq_key y hq_cert de la celda anterior (Part 3)

# ── 1. HQ prepara el firmware y el manifiesto ──
firmware = os.urandom(4096)  # simulamos un fragmento de firmware
firmware_hash = hashlib.sha256(firmware).digest()

manifest = json.dumps({
    'version': '2.4.1',
    'model': 'AutoDrive-X1',
    'sha256': firmware_hash.hex(),
    'timestamp': '2026-04-06T12:00:00Z',
}, sort_keys=True).encode()

# ── 2. HQ firma el manifiesto con su clave privada ──
signature = hq_key.sign(
    manifest,
    padding.PKCS1v15(),
    hashes.SHA256(),
)
print('=== FIRMA DEL MANIFIESTO ===')
print(f'Firmware SHA-256 : {firmware_hash.hex()[:64]}')
print(f'Firma (hex)      : {signature.hex()[:64]}...')

# ── 3. Vehiculo verifica la firma con la clave publica de HQ ──
hq_public_key = hq_cert.public_key()
try:
    hq_public_key.verify(signature, manifest, padding.PKCS1v15(), hashes.SHA256())
    print('\n[OK] Verificacion exitosa: el manifiesto fue firmado por HQ.')
except Exception as e:
    print(f'\n[FALLO] Verificacion fallida: {e}')

# ── 4. Caso de fallo: firmware alterado ──
tampered_firmware = bytearray(firmware)
tampered_firmware[0] ^= 0xFF  # cambiamos un byte
tampered_hash = hashlib.sha256(tampered_firmware).digest()

tampered_manifest = json.dumps({
    'version': '2.4.1',
    'model': 'AutoDrive-X1',
    'sha256': tampered_hash.hex(),
    'timestamp': '2026-04-06T12:00:00Z',
}, sort_keys=True).encode()

print('\n=== CASO DE FALLO: firmware alterado por un atacante ===')
print(f'Hash original  : {firmware_hash.hex()[:32]}...')
print(f'Hash alterado  : {tampered_hash.hex()[:32]}...')

try:
    hq_public_key.verify(signature, tampered_manifest, padding.PKCS1v15(), hashes.SHA256())
    print('[OK] Verificacion exitosa (esto NO deberia pasar)')
except Exception:
    print('[RECHAZADO] La firma no coincide con el manifiesto alterado.')
    print('           El vehiculo rechaza la actualizacion correctamente.')

=== FIRMA DEL MANIFIESTO ===
Firmware SHA-256 : 930b73a3342f726e43285f102257896aa3719ed958f3423b5594b19ddaec8462
Firma (hex)      : 64c3f25969237ddd9b34afc64a9ac52a8794bdb13e8c69a48e728d8c846106b7...

[OK] Verificacion exitosa: el manifiesto fue firmado por HQ.

=== CASO DE FALLO: firmware alterado por un atacante ===
Hash original  : 930b73a3342f726e43285f102257896a...
Hash alterado  : 017955a4d1b2b47be665159c3b9c0ceb...
[RECHAZADO] La firma no coincide con el manifiesto alterado.
           El vehiculo rechaza la actualizacion correctamente.


## Parte 5. System Design Challenge

**Objetivo.** Integrar RSA, SHA-256, certificados y firmas digitales en un protocolo de actualizacion segura, escalable y justificable para la flota AutoDrive.

### Decision de diseno

Se propone un sistema hibrido porque el firmware es grande y no conviene cifrar 2 GB con criptografia asimetrica. La idea central es separar funciones:

- **Criptografia asimetrica:** protege una clave de sesion pequena y autentica identidades.
- **Criptografia simetrica:** cifra eficientemente el firmware grande.
- **Hashes y firmas:** garantizan integridad, autenticacion y no repudio.

Para concretar el diseno, supondremos que en fabrica cada vehiculo recibe la Root CA de confianza y registra su clave publica con AutoDrive HQ. Asi, HQ puede cifrar secretos especificamente para ese vehiculo.

En notacion compacta, la proteccion principal del paquete puede expresarse como

$$
C_K = \mathrm{Enc}_{pk_V}(K_s), \qquad (C_F, tag) = \mathrm{AES\mbox{-}GCM}_{K_s}(F, nonce).
$$

Aqui $K_s$ es la clave de sesion AES, $F$ es el firmware y $C_K$ representa el cifrado asimetrico de la clave de sesion, por ejemplo con RSA-OAEP.

### A. Protocolo seguro de actualizacion paso a paso

1. **Provisionamiento inicial en fabrica.** El vehiculo almacena la Root CA confiable y registra en HQ su clave publica o un certificado propio del vehiculo.
2. **Preparacion del release.** HQ genera el archivo de firmware y calcula su hash SHA-256.
3. **Creacion del manifiesto.** HQ construye un manifiesto con version, hash del firmware, modelo compatible, timestamp y politica de instalacion.
4. **Firma del manifiesto.** HQ firma el manifiesto con su clave privada de firma. Esta firma sera la evidencia de autorizacion oficial de esa version.
5. **Generacion de clave de sesion.** HQ crea una clave aleatoria de sesion AES-256 y un nonce para AES-GCM.
6. **Cifrado del firmware.** HQ cifra el firmware con AES-GCM usando la clave de sesion. Esto protege la confidencialidad y produce un tag de autenticacion del cifrado.
7. **Proteccion de la clave de sesion.** HQ cifra la clave AES con la clave publica del vehiculo, idealmente mediante RSA-OAEP.
8. **Envio del paquete.** HQ envia al vehiculo: certificado de HQ, cadena intermedia necesaria, manifiesto firmado, clave de sesion cifrada, nonce de AES-GCM, firmware cifrado y tag.
9. **Validacion de certificados.** El vehiculo verifica que el certificado de HQ encadena correctamente hasta la Root CA instalada en fabrica.
10. **Verificacion de firma.** El vehiculo usa la clave publica de HQ contenida en el certificado validado para verificar la firma del manifiesto.
11. **Recuperacion de la clave de sesion.** El vehiculo descifra la clave AES con su clave privada.
12. **Descifrado del firmware.** El vehiculo usa AES-GCM para descifrar el firmware y validar el tag de autenticacion.
13. **Comprobacion final de integridad.** El vehiculo recalcula SHA-256 del firmware descifrado y lo compara con el hash autenticado dentro del manifiesto firmado.
14. **Control de instalacion y registro.** Si todo coincide, el vehiculo instala la actualizacion y registra version, hash y evidencia de firma para auditoria posterior.

Si denotamos por $H$ a SHA-256 y por $h_{M_f}$ al hash almacenado en el manifiesto firmado, el vehiculo solo acepta la actualizacion si

$$
\mathrm{Verify}_{pk_{HQ}}(M_f, \sigma) = \text{valida} \quad \text{y} \quad H(F) = h_{M_f}.
$$

### Mini conclusion

El protocolo separa claramente confidencialidad, integridad y autenticacion. Ningun mecanismo trabaja solo: el certificado autentica la clave publica, la firma autentica la autorizacion del release, RSA-OAEP protege la clave de sesion y AES-GCM protege el contenido del firmware.

### B. Tabla de requerimientos y mecanismos criptograficos

| Requirement | How the design satisfies it |
| --- | --- |
| **1. Secure Key Exchange** | HQ genera una clave AES de sesion y la cifra con la clave publica del vehiculo usando criptografia asimetrica, por ejemplo RSA-OAEP. Solo el vehiculo, con su clave privada, puede recuperarla. |
| **2. Confidentiality** | El firmware completo se cifra con AES-256-GCM usando la clave de sesion. Un competidor que intercepte el trafico vera solo ciphertext. |
| **3. Integrity** | AES-GCM verifica integridad del ciphertext mediante su tag, y ademas el vehiculo recalcula SHA-256 del firmware y lo compara con el hash autenticado en el manifiesto firmado. |
| **4. Authentication** | El vehiculo valida el certificado X.509 de HQ hasta una Root CA confiable y luego verifica la firma digital del manifiesto con la clave publica de HQ. |
| **5. Non-Repudiation** | La firma del manifiesto fue generada con la clave privada de HQ y queda asociada a una version especifica del firmware, su hash y sus metadatos. Esto constituye evidencia tecnica de autorizacion. |

### C. Diagrama de interaccion en formato Mermaid

Si tu visor de notebooks no renderiza Mermaid, el diagrama ASCII de la siguiente subseccion sigue disponible como respaldo.

```mermaid
flowchart LR
    HQ["AutoDrive HQ"]
    V["Vehicle"]
    X509["Validacion de certificado X.509"]
    SIG["Verificacion de firma del manifiesto"]
    KEY["Descifrado de clave AES de sesion"]
    DEC["Descifrado del firmware con AES-GCM"]
    HASH["Recalculo de SHA-256 y comparacion final"]
    INSTALL["Instalacion autorizada"]

    HQ -->|Genera hash del firmware| HQ
    HQ -->|Crea y firma el manifiesto| HQ
    HQ -->|Genera clave AES de sesion y nonce| HQ
    HQ -->|Cifra firmware con AES-GCM| HQ
    HQ -->|Cifra clave AES con la clave publica del vehiculo| V
    HQ -->|Envia certificado HQ y cadena intermedia| V
    HQ -->|Envia manifiesto firmado| V
    HQ -->|Envia nonce, ciphertext y tag| V
    V --> X509 --> SIG --> KEY --> DEC --> HASH --> INSTALL
```

### D. Diagrama adicional en formato ASCII

```text
+--------------------+                                +----------------------+
|    AutoDrive HQ    |                                |       Vehicle        |
+--------------------+                                +----------------------+
| 1. Genera hash SHA-256 del firmware                 |                      |
| 2. Crea manifiesto con version + hash              |                      |
| 3. Firma manifiesto con clave privada de HQ        |                      |
| 4. Genera clave AES de sesion + nonce              |                      |
| 5. Cifra firmware con AES-GCM                      |                      |
| 6. Cifra clave AES con clave publica del vehiculo  |                      |
+--------------------+                                +----------------------+
            |                                                        ^
            | 7. Envia: certificado HQ, manifiesto firmado,          |
            |    clave AES cifrada, nonce, ciphertext, tag           |
            v                                                        |
+--------------------+                                +----------------------+
|    Canal de red    |--------------------------------> Verifica cadena X.509 |
+--------------------+                                | Verifica firma HQ    |
                                                      | Descifra clave AES   |
                                                      | Descifra firmware    |
                                                      | Recalcula SHA-256    |
                                                      | Compara hash final   |
                                                      | Instala si todo vale |
                                                      +----------------------+
```

### Mini conclusion

El protocolo propuesto satisface los cinco requerimientos del enunciado y utiliza un enfoque realista para firmware grande. La combinacion de certificados, firma digital, cifrado asimetrico para la clave de sesion y cifrado simetrico para el archivo permite que AutoDrive proteja tanto la autenticidad como la confidencialidad del proceso.

### E. Simulacion end-to-end del protocolo

El siguiente codigo ejecuta el protocolo completo: desde la preparacion del firmware en HQ hasta la instalacion validada en el vehiculo. Cada paso imprime su resultado para que se pueda seguir el flujo descrito en la seccion A.

In [6]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding
import hashlib, json, os

# Reutilizamos ca_key, ca_cert, hq_key, hq_cert de las celdas anteriores

print('=' * 65)
print('   SIMULACION END-TO-END: Secure Firmware Update Protocol')
print('=' * 65)

# ── Paso 0: Provisionamiento en fabrica ──
# El vehiculo genera su par de claves y registra su clave publica en HQ.
vehicle_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
vehicle_pub = vehicle_key.public_key()
print('\n[FABRICA] Vehiculo provisionado con Root CA y par de claves RSA-2048.')

# ══════════════════════════════════════════════════════════
#   LADO HQ: preparacion del paquete de actualizacion
# ══════════════════════════════════════════════════════════

# Paso 1: Firmware y su hash
firmware = os.urandom(8192)  # simulamos 8 KB de firmware
fw_hash = hashlib.sha256(firmware).digest()
print(f'\n[HQ] Paso 1 - Firmware generado ({len(firmware)} bytes)')
print(f'     SHA-256: {fw_hash.hex()[:48]}...')

# Paso 2: Manifiesto
manifest_dict = {
    'version': '3.1.0',
    'model': 'AutoDrive-X1',
    'sha256': fw_hash.hex(),
    'timestamp': '2026-04-06T15:30:00Z',
}
manifest_bytes = json.dumps(manifest_dict, sort_keys=True).encode()
print(f'[HQ] Paso 2 - Manifiesto creado: v{manifest_dict["version"]}')

# Paso 3: Firma del manifiesto
manifest_signature = hq_key.sign(
    manifest_bytes,
    asym_padding.PKCS1v15(),
    hashes.SHA256(),
)
print(f'[HQ] Paso 3 - Manifiesto firmado con clave privada de HQ')

# Paso 4: Clave de sesion AES-256-GCM
aes_key = AESGCM.generate_key(bit_length=256)
nonce = os.urandom(12)
print(f'[HQ] Paso 4 - Clave AES-256 de sesion generada ({len(aes_key)*8} bits)')

# Paso 5: Cifrado del firmware
aesgcm = AESGCM(aes_key)
encrypted_firmware = aesgcm.encrypt(nonce, firmware, None)
print(f'[HQ] Paso 5 - Firmware cifrado con AES-GCM ({len(encrypted_firmware)} bytes incl. tag)')

# Paso 6: Proteccion de la clave de sesion con la clave publica del vehiculo
encrypted_aes_key = vehicle_pub.encrypt(
    aes_key,
    asym_padding.OAEP(
        mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
print(f'[HQ] Paso 6 - Clave AES cifrada con RSA-OAEP del vehiculo')

# ── Paquete enviado ──
package = {
    'hq_cert': hq_cert,
    'ca_cert': ca_cert,
    'manifest': manifest_bytes,
    'signature': manifest_signature,
    'encrypted_aes_key': encrypted_aes_key,
    'nonce': nonce,
    'encrypted_firmware': encrypted_firmware,
}
print(f'\n[HQ] >>> Paquete enviado al vehiculo >>>')

# ══════════════════════════════════════════════════════════
#   LADO VEHICULO: recepcion y validacion
# ══════════════════════════════════════════════════════════
print(f'\n[VEHICULO] <<< Paquete recibido <<<')

# Paso 7: Validar certificado de HQ contra Root CA
# Verificamos que el certificado de HQ fue firmado por la Root CA
try:
    ca_cert.public_key().verify(
        hq_cert.signature,
        hq_cert.tbs_certificate_bytes,
        asym_padding.PKCS1v15(),
        hq_cert.signature_hash_algorithm,
    )
    print('[VEHICULO] Paso 7 - [OK] Certificado de HQ validado contra Root CA')
except Exception:
    print('[VEHICULO] Paso 7 - [FALLO] Certificado de HQ NO es confiable')
    raise

# Paso 8: Verificar firma del manifiesto
try:
    hq_cert.public_key().verify(
        package['signature'],
        package['manifest'],
        asym_padding.PKCS1v15(),
        hashes.SHA256(),
    )
    print('[VEHICULO] Paso 8 - [OK] Firma del manifiesto verificada')
except Exception:
    print('[VEHICULO] Paso 8 - [FALLO] Firma del manifiesto invalida')
    raise

# Paso 9: Descifrar clave AES
recovered_aes_key = vehicle_key.decrypt(
    package['encrypted_aes_key'],
    asym_padding.OAEP(
        mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
print(f'[VEHICULO] Paso 9 - [OK] Clave AES de sesion recuperada')

# Paso 10: Descifrar firmware
aesgcm_vehicle = AESGCM(recovered_aes_key)
decrypted_firmware = aesgcm_vehicle.decrypt(
    package['nonce'],
    package['encrypted_firmware'],
    None,
)
print(f'[VEHICULO] Paso 10 - [OK] Firmware descifrado ({len(decrypted_firmware)} bytes)')

# Paso 11: Verificar integridad del firmware
received_manifest = json.loads(package['manifest'])
local_hash = hashlib.sha256(decrypted_firmware).hexdigest()
expected_hash = received_manifest['sha256']

if local_hash == expected_hash:
    print(f'[VEHICULO] Paso 11 - [OK] Hash SHA-256 coincide')
else:
    print(f'[VEHICULO] Paso 11 - [FALLO] Hash no coincide')
    raise ValueError('Integrity check failed')

# Paso 12: Instalacion
print(f'\n{"=" * 65}')
print(f'   RESULTADO: Firmware v{received_manifest["version"]} instalado exitosamente')
print(f'   Confidencialidad  : AES-256-GCM')
print(f'   Integridad        : SHA-256 verificado')
print(f'   Autenticacion     : Certificado X.509 + firma digital')
print(f'   Key Exchange      : RSA-OAEP')
print(f'   No Repudio        : Firma del manifiesto con clave privada HQ')
print(f'{"=" * 65}')

   SIMULACION END-TO-END: Secure Firmware Update Protocol

[FABRICA] Vehiculo provisionado con Root CA y par de claves RSA-2048.

[HQ] Paso 1 - Firmware generado (8192 bytes)
     SHA-256: 703793f6e50d878fd167cfe3a5a14e576e07a8af4688f74c...
[HQ] Paso 2 - Manifiesto creado: v3.1.0
[HQ] Paso 3 - Manifiesto firmado con clave privada de HQ
[HQ] Paso 4 - Clave AES-256 de sesion generada (256 bits)
[HQ] Paso 5 - Firmware cifrado con AES-GCM (8208 bytes incl. tag)
[HQ] Paso 6 - Clave AES cifrada con RSA-OAEP del vehiculo

[HQ] >>> Paquete enviado al vehiculo >>>

[VEHICULO] <<< Paquete recibido <<<
[VEHICULO] Paso 7 - [OK] Certificado de HQ validado contra Root CA
[VEHICULO] Paso 8 - [OK] Firma del manifiesto verificada
[VEHICULO] Paso 9 - [OK] Clave AES de sesion recuperada
[VEHICULO] Paso 10 - [OK] Firmware descifrado (8192 bytes)
[VEHICULO] Paso 11 - [OK] Hash SHA-256 coincide

   RESULTADO: Firmware v3.1.0 instalado exitosamente
   Confidencialidad  : AES-256-GCM
   Integridad        : SH

## Conclusion final

**Objetivo.** Cerrar el analisis mostrando como cada componente estudiado aporta una propiedad de seguridad distinta y complementaria.

A lo largo del notebook se mostro que la seguridad del pipeline de firmware de AutoDrive no depende de una unica herramienta criptografica. RSA permite proteger secretos pequenos y sustenta el intercambio seguro de claves; SHA-256 resume y fija la identidad binaria del firmware; los certificados X.509 resuelven el problema de confiar en la clave publica correcta; y las firmas digitales convierten una version concreta del firmware en un artefacto autenticado y atribuible.

La leccion central es arquitectonica: en sistemas criticos, la criptografia debe ensamblarse como una cadena de garantias. Si falta la autenticacion de la clave publica, el sistema puede ser suplantado. Si falta la firma, no hay evidencia fuerte de autorizacion. Si falta el hash, no hay forma eficiente de detectar alteraciones en archivos grandes. Si falta el cifrado simetrico del payload, el firmware puede ser espiado. El protocolo hibrido propuesto para AutoDrive coordina esas piezas y responde de forma coherente a los objetivos de confidencialidad, integridad, autenticacion, intercambio seguro de claves y no repudio.

## Referencias

1. RFC 2986. *PKCS #10: Certification Request Syntax Specification Version 1.7*.
2. RFC 5280. *Internet X.509 Public Key Infrastructure Certificate and Certificate Revocation List (CRL) Profile*.
3. NIST FIPS 180-4. *Secure Hash Standard (SHS)*.
4. RFC 8017. *PKCS #1: RSA Cryptography Specifications*.
5. NIST SP 800-38D. *Recommendation for Block Cipher Modes of Operation: Galois/Counter Mode (GCM) and GMAC*.
6. Menezes, van Oorschot y Vanstone. *Handbook of Applied Cryptography*.